# Python Excel Automation: กรองข้อมูล Excel อัตโนมัติด้วย Python #4

**คำอธิบาย:** เรียนรู้วิธีกรองข้อมูล Excel ด้วย Python

**หัวข้อที่ครอบคลุม:**
1. เปิด Excel workbook และอ่านข้อมูล
2. กรองข้อมูลตามคอลัมน์ที่ระบุ
3. กรองด้วยหลายเงื่อนไข (เงื่อนไข OR)
4. เลือกช่วงด้วย CurrentRegion
5. หาค่า Top N
6. หาตำแหน่งคอลัมน์สำหรับกรอง
7. เขียนข้อมูลที่กรองแล้วกลับไป Excel

> **หมายเหตุ:** วิดีโอต้นฉบับใช้ `.api.AutoFilter()` ซึ่งใช้ได้เฉพาะ Windows (COM)
> Notebook นี้ใช้ **pandas** สำหรับการกรองที่ทำงานได้ทั้งบน macOS และ Windows

In [ ]:
# 📦 ติดตั้ง package ที่จำเป็น (ข้ามถ้าติดตั้งแล้ว)
try:
    import xlwings
    import pandas
    print("✅ ติดตั้ง package แล้ว")
except ImportError:
    %pip install xlwings pandas

✅ Packages already installed.


In [ ]:
# 📚 Import ไลบรารี
# xlwings — เปิดและทำงานกับ Excel workbook ที่เปิดอยู่
# pandas — กรองข้อมูลอย่างมีประสิทธิภาพ (แทนที่ .api.AutoFilter() ที่ใช้ได้เฉพาะ Windows)
import xlwings as xw
import pandas as pd

## 1. เปิด Excel Workbook

In [ ]:
# 📥 ดาวน์โหลดไฟล์ข้อมูลตัวอย่างถ้ายังไม่มี
# ตรวจสอบว่าไฟล์มีอยู่ก่อนเพื่อไม่ต้องดาวน์โหลดซ้ำ
import os
import requests

file_name = "Random Sales Data.xlsx"
url = "https://raw.githubusercontent.com/ddeshar/python-for-excel-manage/main/Random%20Sales%20Data.xlsx"

if os.path.exists(file_name):
    print(f"✅ '{file_name}' มีอยู่แล้ว ข้ามการดาวน์โหลด")
else:
    print(f"📥 ไม่พบ '{file_name}' กำลังดาวน์โหลด...")
    response = requests.get(url)
    response.raise_for_status()
    with open(file_name, "wb") as f:
        f.write(response.content)
    print(f"✅ ดาวน์โหลด '{file_name}' สำเร็จ!")

✅ 'Random Sales Data.xlsx' already exists. Skipping download.


In [ ]:
# 📖 เปิด workbook ด้วย xlwings แล้วอ่านข้อมูลเป็น pandas DataFrame
# xw.Book() เปิดไฟล์ใน Excel (จะเห็น Excel เปิดขึ้น)
# .current_region ดึงข้อมูลต่อเนื่องทั้งหมดเริ่มจาก A1
# .options(pd.DataFrame, header=1) แปลงเป็น DataFrame พร้อมหัวข้อ
file_name = "Random Sales Data.xlsx"
sales = xw.Book(file_name)
sht = sales.sheets[0]
print(f"เปิดแล้ว: {sales.name}")
print(f"Sheet ทั้งหมด: {[s.name for s in sales.sheets]}")

df = sht.range('A1').current_region.options(pd.DataFrame, header=1).value
print(f"\nขนาดข้อมูล: {df.shape}")
print(f"คอลัมน์: {list(df.columns)}")
df.head()

Opened: Random Sales Data.xlsx
Sheets: ['Sheet1', 'Order']

Data shape: (5684, 8)
Columns: ['Location', 'Region', 'Customer', 'Order Date', 'Product', 'Quantity', 'Price', 'Total Sale Amount']


,Location,Region,Customer,Order Date,Product,Quantity,Price,Total Sale Amount
Sales Representative,,,,,,,,
Sara Snyder,Massachusetts,East,Raymond Young,2016-01-01 00:00:00,"Bravo II Megaboss 12-Amp Hard Body Upright, Re...",6.0,12.42,74.52
Sara Snyder,New York,East,Helen Dean,2016-01-01 00:00:00,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",7.0,12.42,86.94
Diane Gonzalez,Washington,West,Shirley Chavez,2016-01-01 00:00:00,Acme Hot Forged Carbon Steel Scissors with Nic...,2.0,16.32,32.64
Sara Snyder,New Jersey,East,Brian Ryan,2016-01-01 00:00:00,Bretford CR4500 Series Slim Rectangular Table,1.0,12.42,12.42
Sara Snyder,New Jersey,East,Benjamin Willis,2016-01-01 00:00:00,Eldon Fold 'N Roll Cart System,3.0,17.83,53.49


## 2. กรองข้อมูลตามคอลัมน์ที่ระบุ
กรองข้อมูลด้วยค่าเดียวในคอลัมน์ (เทียบเท่า Excel AutoFilter)

In [ ]:
# 🔍 กรองค่าเดียว — เทียบเท่ากับ AutoFilter ของ Excel บนคอลัมน์เดียว
# df[df[column] == value] กรองแถวที่คอลัมน์เท่ากับค่าที่ระบุ
# เปลี่ยน 'Region' และ 'East' ให้ตรงกับคอลัมน์ข้อมูลจริงของคุณ
filter_column = 'Region'
filter_value = 'East'

filtered_df = df[df[filter_column] == filter_value]
print(f"กรองแล้ว: {filter_column} = '{filter_value}'")
print(f"แถวที่กรองได้: {len(filtered_df)} จาก {len(df)}")
filtered_df.head(10)

Filter applied: Region = 'East'
Filtered rows: 3597 out of 5684


,Location,Region,Customer,Order Date,Product,Quantity,Price,Total Sale Amount
Sales Representative,,,,,,,,
Sara Snyder,Massachusetts,East,Raymond Young,2016-01-01 00:00:00,"Bravo II Megaboss 12-Amp Hard Body Upright, Re...",6.0,12.42,74.52
Sara Snyder,New York,East,Helen Dean,2016-01-01 00:00:00,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",7.0,12.42,86.94
Sara Snyder,New Jersey,East,Brian Ryan,2016-01-01 00:00:00,Bretford CR4500 Series Slim Rectangular Table,1.0,12.42,12.42
Sara Snyder,New Jersey,East,Benjamin Willis,2016-01-01 00:00:00,Eldon Fold 'N Roll Cart System,3.0,17.83,53.49
Sara Snyder,New York,East,Christina Little,2016-01-01 00:00:00,Newell 322,6.0,53.35,320.10
Randy Watson,New York,East,Ruby Matthews,2016-01-02 00:00:00,DXL Angle-View Binders with Locking Rings by S...,6.0,17.83,106.98
Sara Snyder,New York,East,Christopher Oliver,2016-01-02 00:00:00,Belkin F5C206VTEL 6 Outlet Surge,8.0,12.42,99.36
Randy Watson,Massachusetts,East,Kathryn Fox,2016-01-02 00:00:00,Chromcraft Rectangular Conference Tables,9.0,12.42,111.78
Sara Snyder,Massachusetts,East,Walter Kennedy,2016-01-02 00:00:00,Xerox 1967,1.0,12.42,12.42


## 3. แสดงข้อมูลทั้งหมด (ลบตัวกรอง)
ใช้ DataFrame ต้นฉบับที่ยังไม่ได้กรอง

In [ ]:
# 🔄 แสดงข้อมูลทั้งหมด (ลบตัวกรอง) — แค่ใช้ DataFrame ต้นฉบับที่ยังไม่กรอง
# ใน pandas ตัวกรองสร้าง DataFrame ใหม่ ไม่ได้แก้ไขต้นฉบับ
# ดังนั้น "ลบตัวกรอง" ก็แค่กลับไปใช้ 'df' ต้นฉบับ
print(f"ข้อมูลทั้งหมด: {len(df)} แถว")
df.head(10)

All data: 5684 rows


,Location,Region,Customer,Order Date,Product,Quantity,Price,Total Sale Amount
Sales Representative,,,,,,,,
Sara Snyder,Massachusetts,East,Raymond Young,2016-01-01 00:00:00,"Bravo II Megaboss 12-Amp Hard Body Upright, Re...",6.0,12.42,74.52
Sara Snyder,New York,East,Helen Dean,2016-01-01 00:00:00,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",7.0,12.42,86.94
Diane Gonzalez,Washington,West,Shirley Chavez,2016-01-01 00:00:00,Acme Hot Forged Carbon Steel Scissors with Nic...,2.0,16.32,32.64
Sara Snyder,New Jersey,East,Brian Ryan,2016-01-01 00:00:00,Bretford CR4500 Series Slim Rectangular Table,1.0,12.42,12.42
Sara Snyder,New Jersey,East,Benjamin Willis,2016-01-01 00:00:00,Eldon Fold 'N Roll Cart System,3.0,17.83,53.49
Patrick Graham,Washington,West,Annie Jenkins,2016-01-01 00:00:00,Eldon Expressions Wood and Plastic Desk Access...,6.0,53.35,320.10
Sara Snyder,New York,East,Christina Little,2016-01-01 00:00:00,Newell 322,6.0,53.35,320.10
Patrick Graham,Washington,West,None,2016-01-01 00:00:00,Mitel 5320 IP Phone VoIP phone,9.0,16.32,146.88
Randy Watson,New York,East,Ruby Matthews,2016-01-02 00:00:00,DXL Angle-View Binders with Locking Rings by S...,6.0,17.83,106.98


## 4. กรองด้วยหลายเงื่อนไข (เงื่อนไข OR)
เทียบเท่า AutoFilter กับ `Operator=xlOr`

In [ ]:
# 🔍 หลายเงื่อนไข (OR) — เทียบเท่า AutoFilter กับ xlOr operator
# .isin(list) ตรวจสอบว่าค่าอยู่ใน list หรือไม่ (เหมือน SQL IN)
# ตัวอย่าง: แสดงแถวที่ Region เป็น 'East' หรือ 'West'
filter_column = 'Region'
criteria = ['East', 'West']

filtered_df = df[df[filter_column].isin(criteria)]
print(f"กรองแล้ว: {filter_column} อยู่ใน {criteria}")
print(f"แถวที่กรองได้: {len(filtered_df)} จาก {len(df)}")
filtered_df.head(10)

Filter applied: Region in ['East', 'West']
Filtered rows: 5683 out of 5684


,Location,Region,Customer,Order Date,Product,Quantity,Price,Total Sale Amount
Sales Representative,,,,,,,,
Sara Snyder,Massachusetts,East,Raymond Young,2016-01-01 00:00:00,"Bravo II Megaboss 12-Amp Hard Body Upright, Re...",6.0,12.42,74.52
Sara Snyder,New York,East,Helen Dean,2016-01-01 00:00:00,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",7.0,12.42,86.94
Diane Gonzalez,Washington,West,Shirley Chavez,2016-01-01 00:00:00,Acme Hot Forged Carbon Steel Scissors with Nic...,2.0,16.32,32.64
Sara Snyder,New Jersey,East,Brian Ryan,2016-01-01 00:00:00,Bretford CR4500 Series Slim Rectangular Table,1.0,12.42,12.42
Sara Snyder,New Jersey,East,Benjamin Willis,2016-01-01 00:00:00,Eldon Fold 'N Roll Cart System,3.0,17.83,53.49
Patrick Graham,Washington,West,Annie Jenkins,2016-01-01 00:00:00,Eldon Expressions Wood and Plastic Desk Access...,6.0,53.35,320.10
Sara Snyder,New York,East,Christina Little,2016-01-01 00:00:00,Newell 322,6.0,53.35,320.10
Patrick Graham,Washington,West,None,2016-01-01 00:00:00,Mitel 5320 IP Phone VoIP phone,9.0,16.32,146.88
Randy Watson,New York,East,Ruby Matthews,2016-01-02 00:00:00,DXL Angle-View Binders with Locking Rings by S...,6.0,17.83,106.98


## 5. กรองด้วยเงื่อนไข AND
เทียบเท่า AutoFilter กับ `Operator=xlAnd` บนหลายคอลัมน์

In [17]:
# แสดงชื่อคอลัมน์ทั้งหมดใน DataFrame เป็น list
print(df.columns.tolist())

['Location', 'Region', 'Customer', 'Order Date', 'Product', 'Quantity', 'Price', 'Total Sale Amount']


In [ ]:
# 🔍 เงื่อนไข AND — รวมหลายตัวกรองด้วย & (เครื่องหมาย ampersand)
# แต่ละเงื่อนไขต้องอยู่ในวงเล็บ: (เงื่อนไข1) & (เงื่อนไข2)
# เทียบเท่า AutoFilter กับ xlAnd operator
# ตัวอย่าง: Region = 'East' และ Total Sale Amount > 500
filtered_df = df[(df['Region'] == 'East') & (df['Total Sale Amount'] > 500)]
print(f"กรอง: Region = 'East' AND Total Sale Amount > 500")
print(f"แถวที่กรองได้: {len(filtered_df)} จาก {len(df)}")
filtered_df.head(10)

Filter: Region = 'East' AND Total Sale Amount > 500
Filtered rows: 84 out of 5684


,Location,Region,Customer,Order Date,Product,Quantity,Price,Total Sale Amount
Sales Representative,,,,,,,,
Randy Watson,New York,East,Harold Hunter,2016-01-02 00:00:00,Storex DuraTech Recycled Plastic Frosted Binders,10.0,53.35,533.5
Sara Snyder,New York,East,Virginia Alvarez,2016-01-09 00:00:00,Speck Products Candyshell Flip Case,10.0,53.35,533.5
Sara Snyder,Massachusetts,East,Carl Howard,2016-01-13 00:00:00,Newell 311,10.0,53.35,533.5
Frances Warren,Massachusetts,East,Thomas Robinson,2016-01-15 00:00:00,"Tenex File Box, Personal Filing Tote with Lid,...",10.0,53.35,533.5
Sara Snyder,New York,East,Frances Campbell,2016-01-20 00:00:00,"Belkin 19"" Vented Equipment Shelf, Black",10.0,53.35,533.5
Sara Snyder,New York,East,Kimberly Morris,2016-01-22 00:00:00,"Ampad Gold Fibre Wirebound Steno Books, 6"" x 9...",10.0,53.35,533.5
Sara Snyder,New Jersey,East,David Allen,2016-01-22 00:00:00,Bose SoundLink Bluetooth Speaker,10.0,53.35,533.5
Sara Snyder,New Jersey,East,Arthur Simmons,2016-01-25 00:00:00,Ibico Laser Imprintable Binding System Covers,10.0,53.35,533.5
Sara Snyder,New Jersey,East,Cheryl Howard,2016-01-26 00:00:00,Wilson Jones Custom Binder Spines & Labels,10.0,53.35,533.5


## 6. เลือกช่วงด้วย CurrentRegion

In [ ]:
# 📐 CurrentRegion — เลือกบล็อกข้อมูลต่อเนื่องรอบเซลล์
# คล้ายกับ Ctrl+Shift+End ใน Excel — หาขอบเขตข้อมูลอัตโนมัติ
# .address แสดงช่วงเช่น "$A$1:$F$100"
# .rows.count / .columns.count = จำนวนแถวและคอลัมน์ในช่วง
current_region = sht.range('A1').current_region
print(f"ช่วงข้อมูลปัจจุบัน: {current_region.address}")
print(f"จำนวนแถว: {current_region.rows.count}")
print(f"จำนวนคอลัมน์: {current_region.columns.count}")

Current region: $A$1:$I$5685
Rows: 5685
Columns: 9


## 7. หา 5 ยอดขายสูงสุด
เทียบเท่า AutoFilter กับ `xlTop10Items` operator

In [ ]:
# 🏆 ค่าสูงสุด TOP N — เทียบเท่า AutoFilter กับ xlTop10Items
# df.nlargest(n, column) คืนค่า n แถวที่มีค่าสูงสุด
# ง่ายกว่าวิธี AutoFilter ของ Excel มาก!
sales_column = 'Total Sale Amount'
top_n = 5

top_df = df.nlargest(top_n, sales_column)
print(f"Top {top_n} ตาม '{sales_column}':")
top_df

Top 5 by 'Total Sale Amount':


,Location,Region,Customer,Order Date,Product,Quantity,Price,Total Sale Amount
Sales Representative,,,,,,,,
Randy Watson,New York,East,Harold Hunter,2016-01-02 00:00:00,Storex DuraTech Recycled Plastic Frosted Binders,10.0,53.35,533.5
Sara Snyder,New York,East,Virginia Alvarez,2016-01-09 00:00:00,Speck Products Candyshell Flip Case,10.0,53.35,533.5
Patrick Graham,California,West,Frank Garrett,2016-01-09 00:00:00,Newell Chalk Holder,10.0,53.35,533.5
Sara Snyder,Massachusetts,East,Carl Howard,2016-01-13 00:00:00,Newell 311,10.0,53.35,533.5
Frances Warren,Massachusetts,East,Thomas Robinson,2016-01-15 00:00:00,"Tenex File Box, Personal Filing Tote with Lid,...",10.0,53.35,533.5


## 8. 5 ยอดขายต่ำสุด
เทียบเท่า AutoFilter กับ `xlBottom10Items` operator

In [ ]:
# 📉 ค่าต่ำสุด BOTTOM N — เทียบเท่า AutoFilter กับ xlBottom10Items
# df.nsmallest(n, column) คืนค่า n แถวที่มีค่าต่ำสุด
bottom_df = df.nsmallest(top_n, sales_column)
print(f"Bottom {top_n} ตาม '{sales_column}':")
bottom_df

Bottom 5 by 'Total Sale Amount':


,Location,Region,Customer,Order Date,Product,Quantity,Price,Total Sale Amount
Sales Representative,,,,,,,,
Sara Snyder,New Jersey,East,Brian Ryan,2016-01-01 00:00:00,Bretford CR4500 Series Slim Rectangular Table,1.0,12.42,12.42
Sara Snyder,Massachusetts,East,Walter Kennedy,2016-01-02 00:00:00,Xerox 1967,1.0,12.42,12.42
Diane Gonzalez,Oregon,West,Raymond Matthews,2016-01-03 00:00:00,"Howard Miller 13-3/4"" Diameter Brushed Chrome ...",1.0,12.42,12.42
Randy Watson,Massachusetts,East,William Payne,2016-01-03 00:00:00,Panasonic Kx-TS550,1.0,12.42,12.42
Sara Snyder,Connecticut,East,Carlos Stephens,2016-01-04 00:00:00,Wilson Jones Leather-Like Binders with DublLoc...,1.0,12.42,12.42


## 9. หาตำแหน่งคอลัมน์สำหรับกรอง

In [ ]:
# 🔢 หาตำแหน่งคอลัมน์ — มีประโยชน์เมื่อใช้หมายเลข Field ของ AutoFilter
# list(df.columns) ดึงหัวข้อคอลัมน์ทั้งหมด
# .index(name) + 1 แปลงเป็นตำแหน่งเริ่มจาก 1 (ตามที่ Excel ใช้)
headers = list(df.columns)
print(f"หัวข้อ: {headers}")

target_column = 'Total Sale Amount'
if target_column in headers:
    col_position = headers.index(target_column) + 1  # เริ่มจาก 1 สำหรับ Excel
    print(f"คอลัมน์ '{target_column}' อยู่ตำแหน่ง: {col_position}")
else:
    print(f"ไม่พบคอลัมน์ '{target_column}'")

Headers: ['Location', 'Region', 'Customer', 'Order Date', 'Product', 'Quantity', 'Price', 'Total Sale Amount']
Column 'Sales' not found


## ตารางอ้างอิง AutoFilter Operator

| ค่า Operator | ค่าคงที่ | คำอธิบาย |
|---------------|----------|-------------|
| 1 | xlAnd | AND ของ Criteria1 และ Criteria2 |
| 2 | xlOr | OR ของ Criteria1 และ Criteria2 |
| 3 | xlTop10Items | รายการ Top N |
| 4 | xlBottom10Items | รายการ Bottom N |
| 5 | xlTop10Percent | เปอร์เซ็นต์ Top N |
| 6 | xlBottom10Percent | เปอร์เซ็นต์ Bottom N |
| 7 | xlFilterValues | กรองตามค่า |
| 8 | xlFilterCellColor | สีเซลล์ |
| 9 | xlFilterFontColor | สีตัวอักษร |
| 10 | xlFilterIcon | ไอคอนกรอง |
| 11 | xlFilterDynamic | กรองแบบไดนามิก |

## 10. เขียนข้อมูลที่กรองแล้วกลับไป Excel

In [ ]:
# ✍️ เขียนข้อมูลที่กรองแล้วกลับไป Excel — สร้าง sheet ใหม่พร้อมผลลัพธ์
# 1. กรองข้อมูลด้วย pandas
# 2. เพิ่ม sheet ใหม่ (หรือล้าง sheet ที่มีอยู่)
# 3. เขียน DataFrame ลง sheet ด้วย .options(pd.DataFrame)
# 4. .autofit() ปรับความกว้างคอลัมน์ให้พอดีกับข้อมูล
filtered_df = df[df['Region'] == 'East']

if 'Filtered' not in [s.name for s in sales.sheets]:
    new_sht = sales.sheets.add(name='Filtered', after=sales.sheets[-1])
else:
    new_sht = sales.sheets['Filtered']
    new_sht.clear()

new_sht.range('A1').options(pd.DataFrame, index=False).value = filtered_df
new_sht.autofit(axis='columns')
print(f"✅ เขียน {len(filtered_df)} แถวที่กรองแล้วลง sheet 'Filtered'")

# ปิดเมื่อต้องการ (uncomment)
# sales.close()

✅ Wrote 3597 filtered rows to 'Filtered' sheet


---

## 🎮 Mini Project: กรองข้อมูลขั้นเทพ

ทดสอบการกรองข้อมูลด้วย xlwings + pandas!

> 💡 **คำตอบ:** ดูไฟล์ `answers/11_answers.ipynb`

---

### โจทย์ที่ 1: กรอง Top 10 ยอดขายสูงสุด 🟢
ใช้ไฟล์ `Random Sales Data.xlsx`:
1. เปิดด้วย xlwings + อ่านข้อมูลเป็น DataFrame
2. หา Top 10 แถวที่ยอดขายสูงสุด (`nlargest`)
3. เขียนผลลัพธ์ไปยัง sheet ใหม่ `"Top10"`
4. Autofit คอลัมน์

> 💡 Hint: `df.nlargest(10, 'Total Revenue')`

In [ ]:
# โจทย์ที่ 1: กรอง Top 10 ยอดขายสูงสุด
# เขียนโค้ดของคุณที่นี่ 👇



### โจทย์ที่ 2: กรองหลายเงื่อนไข + เขียนกลับ Excel 🟡
1. กรอง: `Region = "East"` AND `Product Category ใน ["Technology", "Furniture"]`
2. เรียงตาม `Total Revenue` จากมากไปน้อย
3. เขียนไป sheet ใหม่ `"East_Tech_Furn"`
4. Autofit + Print จำนวนแถว

> 💡 Hint: `df[(cond1) & (df[col].isin([...]))].sort_values(col, ascending=False)`

In [ ]:
# โจทย์ที่ 2: กรองหลายเงื่อนไข
# เขียนโค้ดของคุณที่นี่ 👇

